# DAAD Degree Scraper

A webscraper to scrape the degree programmes on the DAAD website and then save them in a dataset folder organised by the locations.

## 1.Setup

### 1.1 Installing Dependencies

In [18]:
!pip install pandas
!pip install requests 
!pip install beautifulsoup4
!pip install openpyxl


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: C:\Users\rehas\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: C:\Users\rehas\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: C:\Users\rehas\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: C:\Users\rehas\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [ ]:
import os
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
import math

Setup complete.


In [79]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/114.0 Safari/537.36",
}

current_time = time.strftime( '%d_%m_%Y', time.gmtime())

output_dir = "datasets"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(f"{output_dir}/bachelor", exist_ok=True)
os.makedirs(f"{output_dir}/master", exist_ok=True)

os.makedirs(f"{output_dir}/bachelor/{current_time}", exist_ok=True)
os.makedirs(f"{output_dir}/master/{current_time}", exist_ok=True)

In [25]:
base_url = "https://www.daad.de/en/studying-in-germany/universities/all-degree-programmes/"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/156.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.daad.de/en/studying-in-germany/universities/all-degree-programmes/",
}

### 1.3 Scraper Function

In [65]:
def scrape_degree_programmes(base_url, headers, lastpage=None):
    all_programmes = []

    # using beautifulsoup to parse the first page and get total count of degree programmes
    try:
        resp = requests.get(f"{base_url}{'&' if '?' in base_url else '?'}hec-p=1&hec-limit=100", headers=headers)
        resp.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Could not access the first page: {e}")
        return df
    
    soup = BeautifulSoup(resp.text, "html.parser")

    count_h2 = soup.find("h2", class_="qa-db-result-count u-size-20 u-font-semibold mb-12 mr-16")
    count_text = count_h2.get_text(strip=True).split()[0].replace(",", "")
    if not count_text.isdigit():
        print("No programmes found.")
        return pd.DataFrame()
    total = int(count_text)
    total_pages = math.ceil(total / 100)
    print(f"Found {total} programmes → {total_pages} pages to scrape.")

    for page in range(1, lastpage+1 if lastpage else total_pages + 1):
        # Reuse the already-parsed soup for page 1, fetch fresh for the rest
        if page > 1:
            try:
                resp = requests.get(f"{base_url}{'&' if '?' in base_url else '?'}hec-p={page}&hec-limit=100", headers=headers)
                resp.raise_for_status()
                soup = BeautifulSoup(resp.text, "html.parser")

            except Exception as e:
                print(f"Could not retrieve page {page}: {e}")
                print("Stopping the scraper and saving the programmes collected so far.")
                break


        print(f"Page {page}/{lastpage if lastpage else total_pages}...")
        items = [li for li in soup.find_all("li") if "qa-db-result-item" in " ".join(li.get("class", []))]
        if not items:
            break

        for li in items:
            h3 = li.find("h3")
            spans = h3.find_all("span") if h3 else []
            university = spans[0].get_text(strip=True) if len(spans) > 0 else None
            programme = spans[1].get_text(strip=True) if len(spans) > 1 else None

            dl = li.find("dl", class_="my-0 u-size-18 row items-grid")
            degree_type, location = None, None
            if dl:
                children = dl.find_all(recursive=False)
                if len(children) >= 3:
                    degree_type = children[0].get_text(strip=True)
                    if degree_type and degree_type.startswith("Degree:"):
                        degree_type = degree_type.replace("Degree:", "").strip()

                    loc_parts = children[2].find_all(recursive=False)
                    if len(loc_parts) >= 2:
                        location = loc_parts[1].get_text(strip=True)

            a = li.find("a", href=True)
            detail_url = "https://www.daad.de" + a["href"] if a else None

            all_programmes.append({
                "University": university,
                "Programme": programme,
                "Degree Type": degree_type,
                "Location": location,
                "Detail URL": detail_url
            })

        time.sleep(.3)


    df = pd.DataFrame(all_programmes)
    df = df.drop_duplicates(subset="Detail URL").reset_index(drop=True)
    print(f"Done. {len(df)} unique programmes collected.")
    return df


In [66]:
#test the function
test = scrape_degree_programmes(base_url, headers, lastpage=3)

Found 22594 programmes → 226 pages to scrape.
Page 1/3...
Page 2/3...
Page 3/3...
Done. 300 unique programmes collected.


In [67]:
test, test.describe(), test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   University   300 non-null    object
 1   Programme    300 non-null    object
 2   Degree Type  300 non-null    object
 3   Location     300 non-null    object
 4   Detail URL   300 non-null    object
dtypes: object(5)
memory usage: 11.8+ KB


(                                   University  \
 0                  Technische Hochschule Köln   
 1    Mittweida University of Applied Sciences   
 2      Leipzig University of Applied Sciences   
 3    University of Applied Sciences Fresenius   
 4                      University of Würzburg   
 ..                                        ...   
 295                     University of Hamburg   
 296       University of Education, Weingarten   
 297      Ludwig Maximilians University Munich   
 298                Humboldt University Berlin   
 299                  University of Regensburg   
 
                                  Programme Degree Type  \
 0          3D Animation for Film and Games      Master   
 1                              3D Printing    Bachelor   
 2      3D Printing and Functional Surfaces    Bachelor   
 3                  Academic Speech Therapy    Bachelor   
 4                  Academic Speech Therapy    Bachelor   
 ..                                     ... 

## 2. Scraping the Degrees

In [68]:
#test the function
programmes = scrape_degree_programmes(base_url, headers, lastpage=None)
programmes.head()

Found 22594 programmes → 226 pages to scrape.
Page 1/226...
Page 2/226...
Page 3/226...
Page 4/226...
Page 5/226...
Page 6/226...
Page 7/226...
Page 8/226...
Page 9/226...
Page 10/226...
Page 11/226...
Page 12/226...
Page 13/226...
Page 14/226...
Page 15/226...
Page 16/226...
Page 17/226...
Page 18/226...
Page 19/226...
Page 20/226...
Page 21/226...
Page 22/226...
Page 23/226...
Page 24/226...
Page 25/226...
Page 26/226...
Page 27/226...
Page 28/226...
Page 29/226...
Page 30/226...
Page 31/226...
Page 32/226...
Page 33/226...
Page 34/226...
Page 35/226...
Page 36/226...
Page 37/226...
Page 38/226...
Page 39/226...
Page 40/226...
Page 41/226...
Page 42/226...
Page 43/226...
Page 44/226...
Page 45/226...
Page 46/226...
Page 47/226...
Page 48/226...
Page 49/226...
Page 50/226...
Page 51/226...
Page 52/226...
Page 53/226...
Page 54/226...
Page 55/226...
Page 56/226...
Page 57/226...
Page 58/226...
Page 59/226...
Page 60/226...
Page 61/226...
Page 62/226...
Page 63/226...
Page 64/226...
Pag

,University,Programme,Degree Type,Location,Detail URL
0,Technische Hochschule Köln,3D Animation for Film and Games,Master,Köln,https://www.daad.de/en/studying-in-germany/uni...
1,Mittweida University of Applied Sciences,3D Printing,Bachelor,Mittweida,https://www.daad.de/en/studying-in-germany/uni...
2,Leipzig University of Applied Sciences,3D Printing and Functional Surfaces,Bachelor,Leipzig,https://www.daad.de/en/studying-in-germany/uni...
3,University of Applied Sciences Fresenius,Academic Speech Therapy,Bachelor,"Idstein, Frankfurt am Main",https://www.daad.de/en/studying-in-germany/uni...
4,University of Würzburg,Academic Speech Therapy,Bachelor,Würzburg,https://www.daad.de/en/studying-in-germany/uni...


## 3. Data Processing

### 3.1 Inspecting the Results

In [69]:
programmes.info(), programmes.describe(), programmes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9900 entries, 0 to 9899
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   University   9900 non-null   object
 1   Programme    9900 non-null   object
 2   Degree Type  9900 non-null   object
 3   Location     9900 non-null   object
 4   Detail URL   9900 non-null   object
dtypes: object(5)
memory usage: 386.8+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9900 entries, 0 to 9899
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   University   9900 non-null   object
 1   Programme    9900 non-null   object
 2   Degree Type  9900 non-null   object
 3   Location     9900 non-null   object
 4   Detail URL   9900 non-null   object
dtypes: object(5)
memory usage: 386.8+ KB


(None,
                    University  Programme Degree Type Location  \
 count                    9900       9900        9900     9900   
 unique                    371       4161          10      396   
 top     University of Cologne  Chemistry    Bachelor   Berlin   
 freq                      159        276        4715      437   
 
                                                Detail URL  
 count                                                9900  
 unique                                               9900  
 top     https://www.daad.de/en/studying-in-germany/uni...  
 freq                                                    1  ,
 None)

In [70]:
# Split by degree type
bachelor_df = programmes[programmes["Degree Type"] == "Bachelor"]
master_df   = programmes[programmes["Degree Type"] == "Master"]

### 3.2 Save the degree programmes

In [80]:
# Save one Excel file per unique location, for both Bachelor and Master
for location, group in bachelor_df.groupby("Location"):
    safe_loc = location.replace("/", "_").replace("\\", "_")
    group.to_excel(f"{output_dir}/bachelor/{current_time}/{safe_loc}.xlsx", index=False)

for location, group in master_df.groupby("Location"):
    safe_loc = location.replace("/", "_").replace("\\", "_")
    group.to_excel(f"{output_dir}/master/{current_time}/{safe_loc}.xlsx", index=False)

print(f"Saved {len(bachelor_df)} bachelor and {len(master_df)} master programmes")
print(f"Bachelor locations: {bachelor_df['Location']}")
print(f"Master locations: {master_df['Location']}")

Saved 4715 bachelor and 4511 master programmes
Bachelor locations: 1                        Mittweida
2                          Leipzig
3       Idstein, Frankfurt am Main
4                         Würzburg
5                        Pforzheim
                   ...            
9885                      Freiburg
9886                      Freiburg
9887                     Wuppertal
9888                     Osnabrück
9889                     Magdeburg
Name: Location, Length: 4715, dtype: object
Master locations: 0             Köln
6       Ingolstadt
7        Stuttgart
10          Weimar
12        Nürnberg
           ...    
9895         Essen
9896         Essen
9897      Dortmund
9898      Dortmund
9899     Paderborn
Name: Location, Length: 4511, dtype: object
